### N-Gram Language Model From Scratch

This notebook implements a basic bigram language model using token counts and conditional probabilities.

<b>Implementation Plan</b>
- Prepare and tokenise a corpus
- Add sentence-boundary tokens
- Count unigrams and bigrams
- Calculate bigram probabilities
- Score sentences
- Demonstrate the zero-frequency problem
- Add Laplace smoothing
- Generate text by sampling

<b>1. Import Dependencies</b>

We use `Counter` to count token and bigram frequencies, and `random` to sample words when generating text.

In [228]:
from collections import Counter
import random

<b> 2. Define the Training Corpus </b>

This small corpus contains a few example sentences that the language model will learn from.

In [229]:
corpus = [
    "Logesh read a novel",
    "Deepak played tennis",
    "Joshua read a book while travelling",
    "Mukilan went to the store"
]

<b> 3. Normalize Text </b>

Convert every sentence to lowercase so that words such as `Read` and `read` are treated as the same token.

In [230]:
corpus = [sentence.lower() for sentence in corpus]

<b> 4. Tokenize the Corpus </b>

Split each sentence into individual word tokens.

In [231]:
corpus = [sentence.split() for sentence in corpus]

<b> 5. Add Sentence-Boundary Tokens </b>

Add `<start>` and `<end>` tokens to each sentence. These help the model learn how sentences begin and end.

In [232]:
corpus = [["<start>"] + sentence + ["<end>"] for sentence in corpus]

<b> 6. Inspect the Processed Corpus </b>

Display the tokenized sentences after adding boundary tokens.

In [233]:
for sentence in corpus: print(sentence)

['<start>', 'logesh', 'read', 'a', 'novel', '<end>']
['<start>', 'deepak', 'played', 'tennis', '<end>']
['<start>', 'joshua', 'read', 'a', 'book', 'while', 'travelling', '<end>']
['<start>', 'mukilan', 'went', 'to', 'the', 'store', '<end>']


<b> 6. Inspect the Processed Corpus </b>

Display the tokenized sentences after adding boundary tokens.

In [234]:
flat_tokens = [token for sentence in corpus for token in sentence]
token_counts = [[token, count] for token, count in Counter(flat_tokens).items()]
token_counts = Counter(flat_tokens)

for item in token_counts: print(item)

<start>
logesh
read
a
novel
<end>
deepak
played
tennis
joshua
book
while
travelling
mukilan
went
to
the
store


<b> 7. Count Unigram Frequencies </b>

Flatten the corpus into one list of tokens and count how often each token appears.

In [235]:
bigrams = [
    (sentence[i], sentence[i + 1])
    for sentence in corpus
    for i in range(len(sentence) - 1)
]

bigram_counts = [[list(pair), count] for pair, count in Counter(bigrams).items()]
bigram_counts = Counter(bigrams)

for item in bigram_counts: print(item)

('<start>', 'logesh')
('logesh', 'read')
('read', 'a')
('a', 'novel')
('novel', '<end>')
('<start>', 'deepak')
('deepak', 'played')
('played', 'tennis')
('tennis', '<end>')
('<start>', 'joshua')
('joshua', 'read')
('a', 'book')
('book', 'while')
('while', 'travelling')
('travelling', '<end>')
('<start>', 'mukilan')
('mukilan', 'went')
('went', 'to')
('to', 'the')
('the', 'store')
('store', '<end>')


<b> 8. Create and Count Bigrams </b>

Create word pairs from each sentence, then count how often every pair occurs. For example, `("read", "a")` is a bigram.

In [236]:
token_counts = Counter(flat_tokens)
bigram_counts = Counter(bigrams)

<b> 8. Create and Count Bigrams </b>

Create word pairs from each sentence, then count how often every pair occurs. For example, `("read", "a")` is a bigram.

In [237]:
def bigram_prob(previous_token, next_token):
    denominator = token_counts[previous_token]

    if denominator == 0:
        return 0.0

    numerator = bigram_counts[(previous_token, next_token)]

    return numerator / denominator

<b >9. Test Individual Bigram Probabilities </b>

Check probabilities for bigrams that appear in the corpus and one that does not.

In [238]:
print("P(book | a)       ->", bigram_prob("a", "book"))
print("P(novel | a)      ->", bigram_prob("a", "novel"))
print("P(a | read)       ->", bigram_prob("read", "a"))
print("P(tennis | played)->", bigram_prob("played", "tennis"))
print("P(store | the)    ->", bigram_prob("the", "store"))
print("P(book | read)    ->", bigram_prob("read", "book"))

P(book | a)       -> 0.5
P(novel | a)      -> 0.5
P(a | read)       -> 1.0
P(tennis | played)-> 1.0
P(store | the)    -> 1.0
P(book | read)    -> 0.0


<b> 10. Calculate Sentence Probability </b>

Calculate a sentence’s probability by multiplying the probabilities of all its consecutive bigrams.

In [239]:
def sentence_prob(sentence_tokens):
    total_prob = 1.0

    for i in range(len(sentence_tokens) - 1):
        prev_token = sentence_tokens[i]
        next_token = sentence_tokens[i + 1]

        p = bigram_prob(prev_token, next_token)
        total_prob *= p

        if total_prob == 0.0:
            return 0.0

    return total_prob

<b> 11. Score Example Sentences </b>

Compare probabilities for sentences that follow patterns learned from the training corpus.

In [240]:
s1 = ["<start>", "logesh", "read", "a", "novel", "<end>"]
print("P(Logesh read a novel)                   ->", sentence_prob(s1))

s2 = ["<start>", "logesh", "read", "a", "book", "while", "travelling", "<end>"]
print("P(Logesh read a book while travelling)   ->", sentence_prob(s2))

s3 = ["<start>", "a", "novel", "read", "<end>"]
print("P(a novel read)                         ->", sentence_prob(s3))

P(Logesh read a novel)                   -> 0.125
P(Logesh read a book while travelling)   -> 0.125
P(a novel read)                         -> 0.0


<b> 12. Add Laplace Smoothing </b>

Laplace smoothing adds `1` to every bigram count. This prevents unseen bigrams from receiving a probability of zero.

In [241]:
V = len(token_counts)

def prob_smoothed(previous_token, next_token):
    numerator = bigram_counts[(previous_token, next_token)] + 1
    denominator = token_counts[previous_token] + V

    return numerator / denominator

def sentence_prob_smoothed(sentence_tokens):
    total_prob = 1.0

    for i in range(len(sentence_tokens) - 1):
        prev_token = sentence_tokens[i]
        next_token = sentence_tokens[i + 1]

        p = prob_smoothed(prev_token, next_token)
        total_prob *= p

    return total_prob

<b> 13. Compare Smoothed and Unsmoothed Scores </b>

Show how smoothing gives a small non-zero probability to a sentence containing unseen word transitions.

In [242]:
unseen_sentence = ["<start>", "a", "novel", "read", "<end>"]

print("Unsmoothed sentence prob:", sentence_prob(unseen_sentence))
print("Smoothed sentence prob:  ", sentence_prob_smoothed(unseen_sentence))

Unsmoothed sentence prob: 0.0
Smoothed sentence prob:   1.196172248803828e-05


<b> 14. Generate a Sentence </b>

Start with `<start>` and repeatedly sample the next word according to the learned bigram probabilities until `<end>` is selected or the maximum length is reached.

In [243]:
def generate_sentence_unsmoothed(max_length=20):
    current_token = "<start>"
    generated_sequence = [current_token]

    vocab = [t for t in token_counts.keys() if t != "<start>"]

    for _ in range(max_length):
        probabilities = [bigram_prob(current_token, next_token) for next_token in vocab]

        if sum(probabilities) == 0:
            break

        next_token = random.choices(vocab, weights=probabilities, k=1)[0]
        generated_sequence.append(next_token)

        if next_token == "<end>":
            break

        current_token = next_token

    words = [t for t in generated_sequence if t not in ("<start>", "<end>")]
    return " ".join(words)

<b> 15. Generate a Reproducible Example </b>

Set a random seed so the generated example is reproducible, then generate one sentence from the model.

In [244]:
random.seed(42)

print("Generated Sentence:")
print(generate_sentence_unsmoothed())

Generated Sentence:
joshua read a novel
